# Introduction to TensorFlow

Welcome to this week's programming assignment! Up until now, you've always used <font color=" fuchsia"> Numpy </font> to build neural networks, but this week you'll explore a deep learning framework that allows you to build neural networks more easily. Machine learning frameworks like <font color=" lightgreen"> TensorFlow, PaddlePaddle, Torch, Caffe, Keras </font> , and many others can speed up your machine learning development significantly. <font color="yellow"> TensorFlow 2.10 </font> has made significant improvements over its predecessor, some of which you'll encounter and implement here!

By the end of this assignment, you'll be able to do the following in TensorFlow 2.3:

* Use `tf.Variable` to modify the state of a variable
* Explain the difference between a variable and a constant
* Apply TensorFlow <font color=" cyan"> decorators </font> to speed up code
* Train a Neural Network on a TensorFlow dataset

Programming frameworks like TensorFlow not only cut down on time spent coding, but can also perform optimizations that speed up the code itself. 

## Table of Contents
- [1- Packages](#1)
    - [1.1 - Checking TensorFlow Version](#1-1)
- [2 - Basic Optimization with GradientTape](#2)
    - [2.1 - Linear Function](#2-1)
        - [Exercise 1 - linear_function](#ex-1)
    - [2.2 - Computing the Sigmoid](#2-2)
        - [Exercise 2 - sigmoid](#ex-2)
    - [2.3 - Using One Hot Encodings](#2-3)
        - [Exercise 3 - one_hot_matrix](#ex-3)
    - [2.4 - Initialize the Parameters](#2-4)
        - [Exercise 4 - initialize_parameters](#ex-4)
- [3 - Building Your First Neural Network in TensorFlow](#3)
    - [3.1 - Implement Forward Propagation](#3-1)
        - [Exercise 5 - forward_propagation](#ex-5)
    - [3.2 Compute the Cost](#3-2)
        - [Exercise 6 - compute_cost](#ex-6)
    - [3.3 - Train the Model](#3-3)
- [4 - Bibliography](#4)

<a name='1'></a>
## 1 - Packages

In [1]:
import h5py              # for handling HDF5 files 
import numpy as np    
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.python.framework.ops import EagerTensor                    # for checking if a variable is a tensor 
from tensorflow.python.ops.resource_variable_ops import ResourceVariable   # for checking if a variable is a resource variable
import time

<a name='1-1'></a>
### 1.1 - Checking TensorFlow Version 

You will be using v2.3 for this assignment, for maximum speed and efficiency.

In [2]:
tf.__version__ # check TensorFlow version

'2.10.1'

<a name='2'></a>
## 2 - Basic Optimization with GradientTape

The beauty of TensorFlow 2 is in its simplicity. Basically, all you need to do is implement forward propagation through a computational graph. TensorFlow will compute the derivatives for you, by moving backwards through the graph recorded with `GradientTape`. All that's left for you to do then is specify the cost function and optimizer you want to use! 

When writing a TensorFlow program, the main object to get used and transformed is the `tf.Tensor`. These tensors are the TensorFlow equivalent of Numpy arrays, i.e. multidimensional arrays of a given data type that also contain information about the computational graph.



Here you'll call the TensorFlow dataset created on a HDF5 file, which you can use in place of a Numpy array to store your datasets. You can think of this as a TensorFlow data generator! 

You will use the Hand sign data set, that is composed of images with shape 64x64x3.

In [ ]:
train_dataset = h5py.File('datasets/train_signs.h5', "r") # open the HDF5 file in read mode
test_dataset = h5py.File('datasets/test_signs.h5', "r")

In [4]:
train_dataset.keys()

<KeysViewHDF5 ['list_classes', 'train_set_x', 'train_set_y']>

##### Research the H5py dataformat to understand how to use it.

In [ ]:
# from QWen Code Open
import h5py

# 打开文件
with h5py.File('datasets/train_signs.h5', 'r') as f:
    # 列出所有键
    print("键:", list(f.keys()))
    
    # 对于每个键，打印形状和数据类型, for loop for 1st key, 2nd key, 3rd key 
    for key in f.keys():
        print(f"{key}: 形状={f[key].shape}, 数据类型={f[key].dtype}")
        
        # 打印前几个样本
        # print(f"{key} 的样本数据:")
        dataset = f[key]
        if dataset.size > 0:
            print(f"{key} 的维度: ", dataset.ndim) 
            if dataset.ndim == 1:
                print(dataset[:min(5, len(dataset))])
            elif dataset.ndim == 2:
                print(dataset[:min(3, dataset.shape[0]), :min(5, dataset.shape[1])])
            elif dataset.ndim == 3:
                print(dataset[0, :min(5, dataset.shape[1]), :min(5, dataset.shape[2])])
        print()

键: ['list_classes', 'train_set_x', 'train_set_y']
list_classes: 形状=(6,), 数据类型=int64
list_classes 的维度:  1
[0 1 2 3 4]

train_set_x: 形状=(1080, 64, 64, 3), 数据类型=uint8
train_set_x 的维度:  4

train_set_y: 形状=(1080,), 数据类型=int64
train_set_y 的维度:  1
[5 0 2 5 2]



In [8]:
train_dataset['train_set_x']

<HDF5 dataset "train_set_x": shape (1080, 64, 64, 3), type "|u1">

数据类型 (type): |u1  u1 表示无符号8位整数（unsigned 8-bit integer）范围是 0 到 255
通常用于存储图像像素值

In [ ]:
import h5py
import numpy as np

# 打开 HDF5 文件
with h5py.File('datasets/train_signs.h5', 'r') as f:
    # 获取数据集
    train_set_x = f['train_set_x']
    
    print(f"数据集形状: {train_set_x.shape}")
    print(f"数据类型: {train_set_x.dtype}")
    print(f"数据范围: {np.min(train_set_x)} - {np.max(train_set_x)}")
    
    # 查看第一个图像的前几行和列
    print(f"第一个图像的前 5x5 像素 (R 通道):")
    print(train_set_x[0, :5, :5, 0])  # 显示第一个图像的前 5x5 像素的红色通道
    # 0 表示红色通道, 1 表示绿色通道, 2 表示蓝色通道, same as regular python indexing start from 0
    
    # 查看数据的一些统计信息
    print(f"数据均值: {np.mean(train_set_x)}")
    print(f"数据标准差: {np.std(train_set_x)}")

数据集形状: (1080, 64, 64, 3)
数据类型: uint8
数据范围: 4 - 244
第一个图像的前 5x5 像素 (R 通道):
[[227 227 227 227 227]
 [227 227 228 227 227]
 [227 227 227 227 226]
 [227 227 226 226 226]
 [226 227 226 225 226]]
数据均值: 181.63262615439334
数据标准差: 50.5402466775753


In [10]:
# tf.data.Dataset.from_tensor_slices( list_or_numpy_array ) creates TensorFlow Datasets
x_train = tf.data.Dataset.from_tensor_slices(train_dataset['train_set_x'])
y_train = tf.data.Dataset.from_tensor_slices(train_dataset['train_set_y'])

x_test = tf.data.Dataset.from_tensor_slices(test_dataset['test_set_x'])
y_test = tf.data.Dataset.from_tensor_slices(test_dataset['test_set_y'])

In [6]:
type(x_train)

tensorflow.python.data.ops.dataset_ops.TensorSliceDataset

Since TensorFlow Datasets are generators, you can't access directly the contents unless you iterate over them in a for loop, or by explicitly creating a Python iterator using `iter` and consuming its
elements using `next`. Also, you can inspect the `shape` and `dtype` of each element using the `element_spec` attribute.

In [7]:
print(x_train.element_spec)

TensorSpec(shape=(64, 64, 3), dtype=tf.uint8, name=None)


In [8]:
print(next(iter(x_train)))

tf.Tensor(
[[[227 220 214]
  [227 221 215]
  [227 222 215]
  ...
  [232 230 224]
  [231 229 222]
  [230 229 221]]

 [[227 221 214]
  [227 221 215]
  [228 221 215]
  ...
  [232 230 224]
  [231 229 222]
  [231 229 221]]

 [[227 221 214]
  [227 221 214]
  [227 221 215]
  ...
  [232 230 224]
  [231 229 223]
  [230 229 221]]

 ...

 [[119  81  51]
  [124  85  55]
  [127  87  58]
  ...
  [210 211 211]
  [211 212 210]
  [210 211 210]]

 [[119  79  51]
  [124  84  55]
  [126  85  56]
  ...
  [210 211 210]
  [210 211 210]
  [209 210 209]]

 [[119  81  51]
  [123  83  55]
  [122  82  54]
  ...
  [209 210 210]
  [209 210 209]
  [208 209 209]]], shape=(64, 64, 3), dtype=uint8)


In [9]:
for element in x_train:
    print(element)
    break

tf.Tensor(
[[[227 220 214]
  [227 221 215]
  [227 222 215]
  ...
  [232 230 224]
  [231 229 222]
  [230 229 221]]

 [[227 221 214]
  [227 221 215]
  [228 221 215]
  ...
  [232 230 224]
  [231 229 222]
  [231 229 221]]

 [[227 221 214]
  [227 221 214]
  [227 221 215]
  ...
  [232 230 224]
  [231 229 223]
  [230 229 221]]

 ...

 [[119  81  51]
  [124  85  55]
  [127  87  58]
  ...
  [210 211 211]
  [211 212 210]
  [210 211 210]]

 [[119  79  51]
  [124  84  55]
  [126  85  56]
  ...
  [210 211 210]
  [210 211 210]
  [209 210 209]]

 [[119  81  51]
  [123  83  55]
  [122  82  54]
  ...
  [209 210 210]
  [209 210 209]
  [208 209 209]]], shape=(64, 64, 3), dtype=uint8)


There's one more additional difference between TensorFlow datasets and Numpy arrays: If you need to transform one, you would invoke the `map` method to apply the function passed as an argument to each of the elements.

In [10]:
def normalize(image):
    """
    Transform an image into a tensor of shape (64 * 64 * 3, 1)
    and normalize its components.
    
    Arguments
    image - Tensor.
    
    Returns: 
    result -- Transformed tensor 
    """
    image = tf.cast(image, tf.float32) / 256.0
    image = tf.reshape(image, [-1,1])
    return image

In [11]:
new_train = x_train.map(normalize)
new_test = x_test.map(normalize)

In [12]:
new_train.element_spec

TensorSpec(shape=(12288, 1), dtype=tf.float32, name=None)

In [13]:
print(next(iter(new_train)))

tf.Tensor(
[[0.88671875]
 [0.859375  ]
 [0.8359375 ]
 ...
 [0.8125    ]
 [0.81640625]
 [0.81640625]], shape=(12288, 1), dtype=float32)


<a name='2-1'></a>
### 2.1 - Linear Function

Let's begin this programming exercise by computing the following equation: $Y = WX + b$, where $W$ and $X$ are random matrices and b is a random vector. 

<a name='ex-1'></a>
### Exercise 1 - linear_function

Compute $WX + b$ where $W, X$, and $b$ are drawn from a random normal distribution. W is of shape (4, 3), X is (3,1) and b is (4,1). As an example, this is how to define a constant X with the shape (3,1):
```python
X = tf.constant(np.random.randn(3,1), name = "X")

```
Note that the difference between `tf.constant` and `tf.Variable` is that you can modify the state of a `tf.Variable` but cannot change the state of a `tf.constant`.

You might find the following functions helpful: 
- tf.matmul(..., ...) to do a matrix multiplication
- tf.add(..., ...) to do an addition
- np.random.randn(...) to initialize randomly

In [14]:
# GRADED FUNCTION: linear_function

def linear_function():
    np.random.seed(1)
    X = tf.constant(np.random.randn(3, 1), name="X")
    W = tf.constant(np.random.randn(4, 3), name="W")
    b = tf.constant(np.random.randn(4, 1), name="b")
    # TF2 直接运算
    Y = tf.add(tf.matmul(W, X), b)
    return Y

In [15]:
result = linear_function()
print(result)

assert type(result) == EagerTensor, "Use the TensorFlow API"
assert np.allclose(result, [[-2.15657382], [ 2.95891446], [-1.08926781], [-0.84538042]]), "Error"
print("\033[92mAll test passed")


tf.Tensor(
[[-2.15657382]
 [ 2.95891446]
 [-1.08926781]
 [-0.84538042]], shape=(4, 1), dtype=float64)
All test passed


**Expected Output**: 

```
result = 
[[-2.15657382]
 [ 2.95891446]
 [-1.08926781]
 [-0.84538042]]
```

<a name='2-2'></a>
### 2.2 - Computing the Sigmoid 
Amazing! You just implemented a linear function. TensorFlow offers a variety of commonly used neural network functions like `tf.sigmoid` and `tf.softmax`.

For this exercise, compute the sigmoid of z. 

In this exercise, you will: Cast your tensor to type `float32` using `tf.cast`, then compute the sigmoid using `tf.keras.activations.sigmoid`. 

<a name='ex-2'></a>
### Exercise 2 - sigmoid

Implement the sigmoid function below. You should use the following: 

- `tf.cast("...", tf.float32)`
- `tf.keras.activations.sigmoid("...")`

In [16]:
# GRADED FUNCTION: sigmoid

def sigmoid(z):
    # 强制转换为 float32 以适配 3060 加速
    z = tf.cast(z, tf.float32)
    a = tf.keras.activations.sigmoid(z)
    return a

**Expected Output**: 
<table>
<tr> 
<td>
type
</td>
<td>
class 'tensorflow.python.framework.ops.EagerTensor'
</td>
</tr><tr> 
<td>
dtype
</td>
<td>
"dtype: 'float32'
</td>
</tr>
<tr> 
<td>
Sigmoid(-1)
</td>
<td>
0.2689414
</td>
</tr>
<tr> 
<td>
Sigmoid(0)
</td>
<td>
0.5
</td>
</tr>
<tr> 
<td>
Sigmoid(12)
</td>
<td>
0.999994
</td>
</tr> 

</table> 

<a name='2-3'></a>
### 2.3 - Using One Hot Encodings

Many times in deep learning you will have a $Y$ vector with numbers ranging from $0$ to $C-1$, where $C$ is the number of classes. If $C$ is for example 4, then you might have the following y vector which you will need to convert like this:


<img src="images/onehot.png" style="width:600px;height:150px;">

This is called "one hot" encoding, because in the converted representation, exactly one element of each column is "hot" (meaning set to 1). To do this conversion in numpy, you might have to write a few lines of code. In TensorFlow, you can use one line of code: 

- [tf.one_hot(labels, depth, axis=0)](https://www.tensorflow.org/api_docs/python/tf/one_hot)

`axis=0` indicates the new axis is created at dimension 0

<a name='ex-3'></a>
### Exercise 3 - one_hot_matrix

Implement the function below to take one label and the total number of classes $C$, and return the one hot encoding in a column whise matrix. Use `tf.one_hot()` to do this, and `tf.reshape()` to reshape your one hot tensor! 

- `tf.reshape(tensor, shape)`

In [17]:
def one_hot_matrix(labels, C):
    # TF2 语法简洁化，直接返回结果
    return tf.one_hot(labels, depth=C, axis=0)
# 修改点：添加 tf.reshape(..., [-1]) 确保输出是 (6,) 的向量
# 这样后面在 stack 时才会得到正确的 (6, 样本数) 矩阵
C = 6
new_y_test = y_test.map(lambda x: tf.reshape(one_hot_matrix(x, C), [-1]))
new_y_train = y_train.map(lambda x: tf.reshape(one_hot_matrix(x, C), [-1]))

<a name='2-4'></a>
### 2.4 - Initialize the Parameters 

Now you'll initialize a vector of numbers between zero and one. The function you'll be calling is `tf.keras.initializers.GlorotNormal`, which draws samples from a truncated normal distribution centered on 0, with `stddev = sqrt(2 / (fan_in + fan_out))`, where `fan_in` is the number of input units and `fan_out` is the number of output units, both in the weight tensor. 

To initialize with zeros or ones you could use `tf.zeros()` or `tf.ones()` instead. 

<a name='ex-4'></a>
### Exercise 4 - initialize_parameters

Implement the function below to take in a shape and to return an array of numbers between -1 and 1. 

 - `tf.keras.initializers.GlorotNormal(seed=1)`
 - `tf.Variable(initializer(shape=())`

In [18]:
def initialize_parameters():
    # 设定随机种子
    tf.random.set_seed(1)
    
    # 这里的初始化器很重要
    initializer = tf.keras.initializers.GlorotNormal(seed=1)

    # --- 必须确保这里使用的是 tf.Variable ---
    W1 = tf.Variable(initializer(shape=(25, 12288)))
    b1 = tf.Variable(tf.zeros(shape=(25, 1)))
    W2 = tf.Variable(initializer(shape=(12, 25)))
    b2 = tf.Variable(tf.zeros(shape=(12, 1)))
    W3 = tf.Variable(initializer(shape=(6, 12)))
    b3 = tf.Variable(tf.zeros(shape=(6, 1)))

    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}
    
    return parameters

<a name='3'></a>
## 3 - Building Your First Neural Network in TensorFlow

In this part of the assignment you will build a neural network using TensorFlow. Remember that there are two parts to implementing a TensorFlow model:

- Implement forward propagation
- Retrieve the gradients and train the model

Let's get into it!

<a name='3-1'></a>
### 3.1 - Implement Forward Propagation 

One of TensorFlow's great strengths lies in the fact that you only need to implement the forward propagation function. 

Here, you'll use a TensorFlow decorator, `@tf.function`, which builds a  computational graph to execute the function. `@tf.function` is polymorphic, which comes in very handy, as it can support arguments with different data types or shapes, and be used with other languages, such as Python. This means that you can use data dependent control flow statements.

When you use `@tf.function` to implement forward propagation, the computational graph is activated, which keeps track of the operations. This is so you can calculate your gradients with backpropagation.

<a name='ex-5'></a>
### Exercise 5 - forward_propagation

Implement the `forward_propagation` function.

**Note** Use only the TF API. 

- tf.math.add
- tf.linalg.matmul
- tf.keras.activations.relu


<a name='3-3'></a>
### 3.3 - Train the Model

Let's talk optimizers. You'll specify the type of optimizer in one line, in this case `tf.keras.optimizers.Adam` (though you can use others such as SGD), and then call it within the training loop. 

Notice the `tape.gradient` function: this allows you to retrieve the operations recorded for automatic differentiation inside the `GradientTape` block. Then, calling the optimizer method `apply_gradients`, will apply the optimizer's update rules to each trainable parameter. At the end of this assignment, you'll find some documentation that explains this more in detail, but for now, a simple explanation will do. ;) 


Here you should take note of an important extra step that's been added to the batch training process: 

- `tf.Data.dataset = dataset.prefetch(8)` 

What this does is prevent a memory bottleneck that can occur when reading from disk. `prefetch()` sets aside some data and keeps it ready for when it's needed. It does this by creating a source dataset from your input data, applying a transformation to preprocess the data, then iterating over the dataset the specified number of elements at a time. This works because the iteration is streaming, so the data doesn't need to fit into the memory. 

In [19]:
def forward_propagation(X, parameters):
    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']
    W3 = parameters['W3']
    b3 = parameters['b3']
    
    # 注意：在 TF2 中使用 tf.matmul 而不是 np.dot
    Z1 = tf.add(tf.matmul(W1, X), b1)
    A1 = tf.nn.relu(Z1)
    Z2 = tf.add(tf.matmul(W2, A1), b2)
    A2 = tf.nn.relu(Z2)
    Z3 = tf.add(tf.matmul(W3, A2), b3)
    
    return Z3

In [20]:
def compute_cost(logits, labels):
    # 确保 logits 和 labels 维度一致 (Classes, Samples) -> (Samples, Classes)
    # tf.nn.softmax_cross_entropy_with_logits 期望标签维度在最后
    cost = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(
        tf.transpose(labels), tf.transpose(logits), from_logits=True))
    
    return cost

In [21]:
import math
import tensorflow as tf
import numpy as np

def random_mini_batches(X, Y, mini_batch_size = 64, seed = 0):
    m = X.shape[1]
    mini_batches = []
    np.random.seed(seed)

    # 产生随机索引序列
    permutation = list(np.random.permutation(m))

    # 【修改点】使用 tf.gather 替代 X[:, permutation]
    # 这确保了在 RTX 3060 上直接对显存中的 Tensor 进行快速重排
    shuffled_X = tf.gather(X, permutation, axis=1)
    shuffled_Y = tf.gather(Y, permutation, axis=1)

    # 随后的切片操作 (slice) 在 TF2 中是原生支持的
    num_complete_minibatches = math.floor(m / mini_batch_size) 
    for k in range(0, num_complete_minibatches):
        mini_batch_X = shuffled_X[:, k * mini_batch_size : (k + 1) * mini_batch_size]
        mini_batch_Y = shuffled_Y[:, k * mini_batch_size : (k + 1) * mini_batch_size]
        mini_batches.append((mini_batch_X, mini_batch_Y))

    if m % mini_batch_size != 0:
        mini_batch_X = shuffled_X[:, num_complete_minibatches * mini_batch_size : m]
        mini_batch_Y = shuffled_Y[:, num_complete_minibatches * mini_batch_size : m]
        mini_batches.append((mini_batch_X, mini_batch_Y))

    return mini_batches

In [ ]:
def model(X_train, Y_train, X_test, Y_test, learning_rate = 0.0001,
          num_epochs = 1500, minibatch_size = 32, print_cost = True):
    
    optimizer = tf.keras.optimizers.Adam(learning_rate)
    costs = []
    
    # 初始化参数
    parameters = initialize_parameters()
    
    # 提取可训练变量列表
    W1, b1 = parameters['W1'], parameters['b1']
    
    W2, b2 = parameters['W2'], parameters['b2']
    W3, b3 = parameters['W3'], parameters['b3']
    trainable_variables = [W1, b1, W2, b2, W3, b3]

    for epoch in range(num_epochs):
        epoch_cost = 0.
        
        # 分批次训练
        minibatches = random_mini_batches(X_train, Y_train, minibatch_size)
        
        for minibatch in minibatches:
            (minibatch_X, minibatch_Y) = minibatch
            
            with tf.GradientTape() as tape:
                # 1. 前向传播
                Z3 = forward_propagation(minibatch_X, parameters)
                # 2. 计算损失
                cost = compute_cost(Z3, minibatch_Y)
            
            # 3. 计算梯度 (注意：必须传入列表而非字典)
            grads = tape.gradient(cost, trainable_variables)
            
            # 4. 更新参数
            optimizer.apply_gradients(zip(grads, trainable_variables))
            
            epoch_cost += cost
        
        epoch_cost /= len(minibatches)
        if print_cost and epoch % 10 == 0:
            print(f"Cost after epoch {epoch}: {epoch_cost}")
        if print_cost and epoch % 1 == 0:
            costs.append(epoch_cost)

    return parameters

In [23]:
# 1. 提取并转换 Tensor
X_train_tensor = tf.stack([tf.squeeze(x) for x in new_train], axis=1)
Y_train_tensor = tf.stack([y for y in new_y_train], axis=1)
X_test_tensor = tf.stack([tf.squeeze(x) for x in new_test], axis=1)
Y_test_tensor = tf.stack([y for y in new_y_test], axis=1) # 补全这一行

print(f"X_train shape: {X_train_tensor.shape}") # 应为 (12288, 1080)
print(f"Y_test shape: {Y_test_tensor.shape}")   # 应为 (6, 120)

# 2. 开始训练
parameters = model(X_train_tensor, Y_train_tensor, X_test_tensor, Y_test_tensor, num_epochs=200)

X_train shape: (12288, 1080)
Y_test shape: (6, 120)
Cost after epoch 0: 1.7986507415771484
Cost after epoch 10: 1.6439540386199951
Cost after epoch 20: 1.525425910949707
Cost after epoch 30: 1.4177638292312622
Cost after epoch 40: 1.3202645778656006
Cost after epoch 50: 1.2352182865142822
Cost after epoch 60: 1.1616814136505127
Cost after epoch 70: 1.0959528684616089
Cost after epoch 80: 1.0367985963821411
Cost after epoch 90: 0.9834288358688354
Cost after epoch 100: 0.9348353147506714
Cost after epoch 110: 0.8903025984764099
Cost after epoch 120: 0.850567638874054
Cost after epoch 130: 0.8139758706092834
Cost after epoch 140: 0.7814297676086426
Cost after epoch 150: 0.7499657273292542
Cost after epoch 160: 0.7212880849838257
Cost after epoch 170: 0.6955354809761047
Cost after epoch 180: 0.6704937815666199
Cost after epoch 190: 0.6480714678764343


**Expected output**

```
Cost after epoch 0: 0.742591
Cost after epoch 10: 0.614557
Cost after epoch 20: 0.598900
Cost after epoch 30: 0.588907
Cost after epoch 40: 0.579898
...
```

**Congratulations**! You've made it to the end of this assignment, and to the end of this week's material. Amazing work building a neural network in TensorFlow 2.3! 

Here's a quick recap of all you just achieved:

- Used `tf.Variable` to modify your variables
- Applied TensorFlow decorators and observed how they sped up your code
- Trained a Neural Network on a TensorFlow dataset
- Applied batch normalization for a more robust network

You are now able to harness the power of TensorFlow's computational graph to create cool things, faster. Nice! 

<a name='4'></a>
## 4 - Bibliography 

In this assignment, you were introducted to `tf.GradientTape`, which records operations for differentation. Here are a couple of resources for diving deeper into what it does and why: 

Introduction to Gradients and Automatic Differentiation: 
https://www.tensorflow.org/guide/autodiff 

GradientTape documentation:
https://www.tensorflow.org/api_docs/python/tf/GradientTape